In [5]:
import torch 
import torch.nn.functional as F
torch.set_printoptions(threshold=float('inf'))
torch.set_printoptions(sci_mode=False)
# init memory and buffer
mem_A = torch.zeros(31008, 16)
mem_B = torch.zeros(31008, 16)
mem_w = torch.zeros(585, 16)
reg_b = torch.zeros(5, 16)

In [2]:
model = torch.load('Model_May27_0439.pt', map_location='cpu')

In [3]:
def linear_quantize(fp_tensor, bitwidth, scale, zero_point, dtype=torch.int8) -> torch.Tensor:
    """
    linear quantization for single fp_tensor
      from
        fp_tensor = (quantized_tensor - zero_point) * scale
      we have,
        quantized_tensor = int(round(fp_tensor / scale)) + zero_point
    :param tensor: [torch.(cuda.)FloatTensor] floating tensor to be quantized
    :param bitwidth: [int] quantization bit width
    :param scale: [torch.(cuda.)FloatTensor] scaling factor
    :param zero_point: [torch.(cuda.)IntTensor] the desired centroid of tensor values
    :return:
        [torch.(cuda.)FloatTensor] quantized tensor whose values are integers
    """
    assert(fp_tensor.dtype == torch.float)
    assert(isinstance(scale, float) or
           (scale.dtype == torch.float and scale.dim() == fp_tensor.dim()))
    assert(isinstance(zero_point, int) or
           (zero_point.dtype == dtype and zero_point.dim() == fp_tensor.dim()))

    # Step 1: scale the fp_tensor
    scaled_tensor = fp_tensor/scale
    #print('scaled tensor: ', scaled_tensor)
    # Step 2: round the floating value to integer value
    rounded_tensor = torch.round(scaled_tensor)

    rounded_tensor = rounded_tensor.to(dtype)
    #print('rounded_Tensor: ', rounded_tensor)
    # Step 3: shift the rounded_tensor to make zero_point 0
    shifted_tensor = rounded_tensor + zero_point
    #print('shifted_tensor: ', shifted_tensor)
    # Step 4: clamp the shifted_tensor to lie in bitwidth-bit range
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    quantized_tensor = shifted_tensor.clamp_(quantized_min, quantized_max)
    #print('quantized_tensor: ', quantized_tensor)
    return quantized_tensor

def get_quantization_scale_and_zero_point(fp_tensor, bitwidth):
    """
    get quantization scale for single tensor
    :param fp_tensor: [torch.(cuda.)Tensor] floating tensor to be quantized
    :param bitwidth: [int] quantization bit width
    :return:
        [float] scale
        [int] zero_point
    """
    #print(fp_tensor)
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    fp_max = fp_tensor.max().item()
    fp_min = fp_tensor.min().item()

    # hint: one line of code for calculating scale
    scale = (fp_max - fp_min)/(quantized_max - quantized_min)
    if scale == 0:
        scale=1e-9
    
    # hint: one line of code for calculating zero_point
    zero_point = round(quantized_min - fp_min/scale)
    # print("quantized min :", quantized_min)
    # print("quantized max :", quantized_max)
    # print("fp min :", fp_min)
    # print("fp max :", fp_max)
    # print("scale :", scale)
    # print("zero point :", zero_point)

    # clip the zero_point to fall in [quantized_min, quantized_max]
    if zero_point < quantized_min:
        zero_point = quantized_min
    elif zero_point > quantized_max:
        zero_point = quantized_max
    else: # convert from float to int using round()
        zero_point = round(zero_point)
    return scale, int(zero_point)

def linear_quantize_feature(fp_tensor, bitwidth): #diff
    """
    linear quantization for feature tensor
    :param fp_tensor: [torch.(cuda.)Tensor] floating feature to be quantized
    :param bitwidth: [int] quantization bit width
    :return:
        [torch.(cuda.)Tensor] quantized tensor
        [float] scale tensor
        [int] zero point
    """
    scale, zero_point = get_quantization_scale_and_zero_point(fp_tensor, bitwidth)
    quantized_tensor = linear_quantize(fp_tensor, bitwidth, scale, zero_point)
    return quantized_tensor, scale, zero_point

def get_quantized_range(bitwidth):
    quantized_max = (1 << (bitwidth - 1)) - 1
    quantized_min = -(1 << (bitwidth - 1))
    return quantized_min, quantized_max

In [4]:
# image & pixel mask
input_img = torch.load('sample_input.pt')[0:3].permute(1, 0, 2, 3) # R, G, B image with size 100 x 100
pixel_mask = torch.load('sample_input.pt')[3:6].permute(1, 0, 2, 3)
input_img , _, _= linear_quantize_feature(input_img, 8)
pixel_mask, _, _= linear_quantize_feature(pixel_mask, 8)
img_H = 100
img_W = 100
pad_H = 1
pad_W = 1
B, C, H, W = 1, 2, img_H+pad_H*2, img_W+pad_W*2

In [5]:
def padding(memory): # for memory A and B
    address, word = memory.size()
    for addr in range(address):
        # pad vertically
        if (addr%W==0) or (addr%W==W-1): # (addr%102==0) or (addr%102==101)
            memory[addr, :] = torch.zeros(word)
        # pad horizontally
        if (addr >= 0 and addr <= W-1) or ( addr >= (H-1)*W and addr <=H*W) or (addr >= (2*(H-1)*W) and addr <= (2*H*W) ) or ( addr >= 3*(H-1) and addr <= (3*H*W)): # 0~101, 10302~10403, 20604 ~ , 30906~
            memory[addr, :] = torch.zeros(word)
        
            
def padding_after_layer4 (memory):
    address, word = memory.size()
    for addr in range(address):
        # pad vertically
        if (addr%W==0) or ((addr-52)%W==W-1): # (addr%102==0) or (addr-52%102==101)
            memory[addr, :] = torch.zeros(word, dtype=torch.int8)
        # pad horizontally
        if (addr >= 0 and addr <= 52-1) or ( addr >= 2652 and addr <= 2702) or (addr >= 7854 and addr <= 7904) or (addr >= 10506 and addr <= 10556): 
            memory[addr, :] = torch.zeros(word, dtype=torch.int8)

In [ ]:
class init_SRAM_write():
    def __call__(self, img, mask):
        # R
        for j in range(100): # H
            for k in range(100): # W
                mem_A[103+j*102+k, 0] = img[0, 0, j:j+1, k:k+1] #channel 1 = img
                mem_A[103+j*102+k, 1] = mask[0, 0, j:j+1, k:k+1] #channel 2 = pixel mask
        # G
        for j in range(100): # H
            for k in range(100): # W
                mem_A[10405+j*102+k, 0] = img[0, 1, j:j+1, k:k+1] #channel 1 = img
                mem_A[10405+j*102+k, 1] = mask[0, 1, j:j+1, k:k+1] #channel 2 = pixel mask
        # B
        for j in range(100): # H
            for k in range(100): # W
                mem_A[20707+j*102+k, 0] = img[0, 2, j:j+1, k:k+1] #channel 1 = img
                mem_A[20707+j*102+k, 1] = mask[0, 2, j:j+1, k:k+1] #channel 2 = pixel mask
        print('memory A write done') 
        
class init_weight_write():
    def __init__(self,):
        self.conv1_weight = model['shared_block.conv1.weight']
        self.conv1_bias = model['shared_block.conv1.bias']
        self.conv2_weight=model['shared_block.conv2.weight']
        self.conv2_bias=model['shared_block.conv2.bias']
        self.conv3_weight=model['shared_block.conv3.weight']
        self.conv3_bias=model['shared_block.conv3.bias']
        self.conv4_weight=model['shared_block.conv4.weight']
        self.conv4_bias=model['shared_block.conv4.bias']
        self.conv5_weight=model['shared_block.conv5.weight']
        self.conv5_bias=model['shared_block.conv5.bias']
    def __call__(self,):
        mem_w[0:144, 0:1]=self.conv1_weight[:, 0, :, :].reshape(144, 1)
        mem_w[0:144, 1:2]=self.conv1_weight[:, 1, :, :].reshape(144, 1)
        mem_w[144:288]=self.conv2_weight.permute(0,2,3,1).reshape(144, 16)
        mem_w[288:432]=self.conv3_weight.permute(0,2,3,1).reshape(144, 16)
        mem_w[432:576]=self.conv4_weight.permute(0,2,3,1).reshape(144, 16)
        mem_w[576:585]=self.conv5_weight.permute(0,2,3,1).reshape(9, 16)
        reg_b[0, :]=self.conv1_bias
        reg_b[1, :]=self.conv2_bias
        reg_b[2, :]=self.conv3_bias
        reg_b[3, :]=self.conv4_bias
        reg_b[4, 0:1]=self.conv5_bias
        print('weight write done')
        
        
class layer1 ():
    def __call__(self,):
        x_r = mem_A[0:10404, 0:2].reshape(-1, 102, 2).permute(2, 0, 1).unsqueeze(0)
        x_g = mem_A[10302: 20706, 0:2].reshape(-1, 102, 2).permute(2, 0, 1).unsqueeze(0)
        x_b = mem_A[20604:31009, 0:2].reshape(-1, 102, 2).permute(2, 0, 1).unsqueeze(0)
        weight=mem_w[0:144, 0:2].reshape(16, 3, 3, 2).permute(0,3,1,2)
        bias=reg_b[0, :]
        y_r = bias_relu(F.conv2d(x_r, weight), bias, layer_state=1, relu_en=True, params=params)
        y_g = bias_relu(F.conv2d(x_g, weight), bias, layer_state=1, relu_en=True, params=params)
        y_b = bias_relu(F.conv2d(x_b, weight), bias, layer_state=1, relu_en=True, params=params)
        y_r=F.pad(y_r, (1,1,1,1))
        y_g=F.pad(y_g, (1,1,1,1))
        y_b=F.pad(y_b, (1,1,1,1)) # 1, 16, 102, 102
        
        mem_B[0:10404, :]=y_r.squeeze().permute(1,2,0).reshape(-1, 16)
        mem_B[10302:20706, :]=y_g.squeeze().permute(1,2,0).reshape(-1, 16)
        mem_B[20604:31009, :]=y_b.squeeze().permute(1,2,0).reshape(-1, 16)
        print("layer 1 done")
        
class layer2 ():
    def __call__(self,):
        x_r = mem_B[0:10404, :].reshape(-1, 102, 16).permute(2,0,1).unsqueeze(0)
        x_g = mem_B[10302: 20706, :].reshape(-1, 102, 16).permute(2,0,1).unsqueeze(0)
        x_b = mem_B[20604:31009, :].reshape(-1, 102, 16).permute(2,0,1).unsqueeze(0)
        weight=mem_w[144:288].reshape(16, 3, 3, 16).permute(0, 3, 1, 2)
        bias=reg_b[1, :]
        y_r = bias_relu(F.conv2d(x_r, weight), bias, layer_state=2, relu_en=True, params=params)
        y_g = bias_relu(F.conv2d(x_g, weight), bias, layer_state=2, relu_en=True, params=params)
        y_b = bias_relu(F.conv2d(x_b, weight), bias, layer_state=2, relu_en=True, params=params)
        y_r=F.pad(y_r, (1,1,1,1))
        y_g=F.pad(y_g, (1,1,1,1))
        y_b=F.pad(y_b, (1,1,1,1)) # 1, 16, 102, 102
        mem_A[0:10404, :]=y_r.squeeze().permute(1,2,0).reshape(-1, 16)
        mem_A[10302:20706, :]=y_g.squeeze().permute(1,2,0).reshape(-1, 16)
        mem_A[20604:31009, :]=y_b.squeeze().permute(1,2,0).reshape(-1, 16)
        print("layer 2 done")
        
class layer3 ():
    def __call__(self,):
        x_r = mem_A[0:10404, :].reshape(-1, 102, 16).permute(2,0,1).unsqueeze(0)
        x_g = mem_A[10302: 20706, :].reshape(-1, 102, 16).permute(2,0,1).unsqueeze(0)
        x_b = mem_A[20604:31009, :].reshape(-1, 102, 16).permute(2,0,1).unsqueeze(0)
        weight=mem_w[288:432].reshape(16, 3, 3, 16).permute(0, 3, 1, 2)
        bias=reg_b[2, :]
        y_r = bias_relu(F.conv2d(x_r, weight), bias, layer_state=3, relu_en=True, params=params)
        y_g = bias_relu(F.conv2d(x_g, weight), bias, layer_state=3, relu_en=True, params=params)
        y_b = bias_relu(F.conv2d(x_b, weight), bias, layer_state=3, relu_en=True, params=params)
        y_r=F.pad(y_r, (1,1,1,1))
        y_g=F.pad(y_g, (1,1,1,1))
        y_b=F.pad(y_b, (1,1,1,1))
        mem_B[0:10404, :]=y_r.squeeze().permute(1,2,0).reshape(-1, 16)
        mem_B[10302:20706, :]=y_g.squeeze().permute(1,2,0).reshape(-1, 16)
        mem_B[20604:31009, :]=y_b.squeeze().permute(1,2,0).reshape(-1, 16)
        print("layer 3 done")
        
class layer4 (): # maxpool
    def __call__(self,):
        x_r = mem_B[0:10404, :].reshape(-1, 102, 16)[1:101, 1:101, :].permute(2,0,1).unsqueeze(0)
        x_g = mem_B[10302: 20706, :].reshape(-1, 102, 16)[1:101, 1:101, :].permute(2,0,1).unsqueeze(0)
        x_b = mem_B[20604:31009, :].reshape(-1, 102, 16)[1:101, 1:101, :].permute(2,0,1).unsqueeze(0)
        print(x_r.shape, x_g.shape, x_b.shape)
        y_r=F.max_pool2d(x_r.float(), kernel_size=(2,4), stride=(2,4)).to(torch.int8) #1, 16, 50, 25
        y_g=F.max_pool2d(x_g.float(), kernel_size=2, stride=2).to(torch.int8)         #1, 16, 50, 50
        y_b=F.max_pool2d(x_b.float(), kernel_size=(2,4), stride=(2,4)).to(torch.int8) #1, 16, 50, 25
        print(y_r.shape, y_g.shape, y_b.shape)
        for i in range(25):
            mem_A[103+i*102:153+i*102, :]=y_r.squeeze().transpose(1,2).permute(1,2,0)[i, :, :]
        
        for i in range(50):
            mem_A[2755+i*102:2805+i*102, :]=y_g.squeeze().transpose(1,2).permute(1,2,0)[i, :, :]
            
        for i in range(25):
            mem_A[7957+i*102:8007+i*102, :]=y_b.squeeze().transpose(1,2).permute(1,2,0)[i, :, :]
        print("layer 4 done")
        
        
class layer5 ():
    def __call__(self,):
        padding_after_layer4(mem_A)
        x_r = torch.zeros(1, 16, 52, 27, dtype=torch.int8)
        for i in range(27):
            x_r[0, :, :, i] = mem_A[0+102*i:52+102*i, :].permute(1, 0).reshape(16, 52).unsqueeze(0)
        x_g = torch.zeros(1, 16, 52, 52, dtype=torch.int8)
        for i in range(52):
            x_g[0, :, :, i] = mem_A[2652+102*i:2704+102*i, :].permute(1, 0).reshape(16, 52).unsqueeze(0)
        x_b = torch.zeros(1, 16, 52, 27, dtype=torch.int8)
        for i in range(27):
            x_b[0, :, :, i] = mem_A[7854+102*i:7906+102*i, :].permute(1, 0).reshape(16, 52).unsqueeze(0)
        weight=mem_w[432:576].reshape(16, 3, 3, 16).permute(0, 3, 1, 2)
        bias=reg_b[3, :]
        y_r=bias_relu(F.conv2d(x_r, weight), bias, layer_state=5, relu_en=True, params=params)
        y_g=bias_relu(F.conv2d(x_g, weight), bias, layer_state=5, relu_en=True, params=params)
        y_b=bias_relu(F.conv2d(x_b, weight), bias, layer_state=5, relu_en=True, params=params)
        print(y_r.shape, y_g.shape, y_b.shape)
        for i in range(25):
            mem_B[103+i*102:153+i*102, :]=y_r.squeeze().permute(1,2,0).transpose(0,1)[i, :, :]
        
        for i in range(50):
            mem_B[2754+i*102:2804+i*102, :]=y_g.squeeze().permute(1,2,0).transpose(0,1)[i, :, :]
            
        for i in range(25):
            mem_B[7956+i*102:8006+i*102, :]=y_b.squeeze().permute(1,2,0).transpose(0,1)[i, :, :]
        print("layer 5 done")
        
class layer6 ():
    def __call__(self,):
        
        padding_after_layer4(mem_B)
        x_r = torch.zeros(1, 16, 52, 27, dtype=torch.int8)
        for i in range(27):
            x_r[0, :, :, i] = mem_B[0+102*i:52+102*i, :].permute(1, 0).reshape(16, 52).unsqueeze(0)
        x_g = torch.zeros(1, 16, 52, 52, dtype=torch.int8)
        for i in range(52):
            x_g[0, :, :, i] = mem_B[2652+102*i:2704+102*i, :].permute(1, 0).reshape(16, 52).unsqueeze(0)
        x_b = torch.zeros(1, 16, 52, 27, dtype=torch.int8)
        for i in range(27):
            x_b[0, :, :, i] = mem_B[7854+102*i:7906+102*i, :].permute(1, 0).reshape(16, 52).unsqueeze(0)
        weight=mem_w[576:585].reshape(1, 3, 3, 16).permute(0, 3, 1, 2)
        bias=reg_b[4, 0:1]
        y_r=bias_relu(F.conv2d(x_r, weight), bias, layer_state=6, relu_en=True, params=params)
        y_g=bias_relu(F.conv2d(x_g, weight), bias, layer_state=6, relu_en=True, params=params)
        y_b=bias_relu(F.conv2d(x_b, weight), bias, layer_state=6, relu_en=True, params=params)
        print(y_r.shape, y_g.shape, y_b.shape)
        for i in range(25):
            mem_A[103+i*102:153+i*102, 0:1]=y_r.squeeze(0).permute(1,2,0).transpose(0,1)[i, :, :]
        
        for i in range(50):
            mem_A[2754+i*102:2804+i*102, 0:1]=y_g.squeeze(0).permute(1,2,0).transpose(0,1)[i, :, :]
            
        for i in range(25):
            mem_A[7956+i*102:8006+i*102, 0:1]=y_b.squeeze(0).permute(1,2,0).transpose(0,1)[i, :, :]
        print("layer 6 done")


In [7]:
def check(memory):
    x_r = memory[0:10404, :].reshape(-1, 102, 16).permute(2, 0, 1)
    x_g = memory[10302: 20706, :].reshape(-1, 102, 16).permute(2, 0, 1)
    x_b = memory[20604:31009, :].reshape(-1, 102, 16).permute(2, 0, 1)
    print(x_r, x_g, x_b)

In [8]:
def check_after_layer4(memory):
    x_r = torch.zeros(1, 1, 52, 27, dtype=torch.int8)
    for i in range(27):
        x_r[0, :, :, i] = memory[0+102*i:52+102*i, 0:1].permute(1, 0).reshape(1, 52).unsqueeze(0)
    x_g = torch.zeros(1, 1, 52, 52, dtype=torch.int8)
    for i in range(52):
        x_g[0, :, :, i] = memory[2652+102*i:2704+102*i, 0:1].permute(1, 0).reshape(1, 52).unsqueeze(0)
    x_b = torch.zeros(1, 1, 52, 27, dtype=torch.int8)
    for i in range(27):
        x_b[0, :, :, i] = memory[7854+102*i:7906+102*i, 0:1].permute(1, 0).reshape(1, 52).unsqueeze(0)
    result_r = x_r[:, :, 1:51, 1:26]
    result_g = x_g[:, :, 1:51, 1:51]
    result_b = x_b[:, :, 1:51, 1:26]
    print(result_r, result_g, result_b)

In [ ]:
import torch

def bias_relu(data_in, bias, layer_state, relu_en, params):
    if not relu_en:
        return torch.zeros(1, dtype=torch.int8)

    # 공통 상수
    WIDTH_BIAS = params['WIDTH_BIAS']
    WIDTH_BIAS_ADDED = params['WIDTH_BIAS_ADDED']
    WIDTH_IN_DATA = params['WIDTH_IN_DATA']
    WIDTH_OUT_DATA = params['WIDTH_OUT_DATA']

    # sign-extend & shift
    sign_bit = data_in >> (WIDTH_IN_DATA - 1)
    data_extended = (sign_bit << (WIDTH_BIAS - WIDTH_IN_DATA)).expand(WIDTH_BIAS - WIDTH_IN_DATA)  # sign extension
    data_extended = (data_extended << WIDTH_IN_DATA) | (data_in & ((1 << WIDTH_IN_DATA) - 1))
    data_extended = data_extended.to(torch.int32)

    # 레이어 파라미터 선택
    key = f'L{layer_state}'
    if key not in params['layers']:
        return torch.zeros(1, dtype=torch.int8)

    p = params['layers'][key]

    EXT_FL = WIDTH_BIAS - p['IN_IL'] - 1
    B_FL   = WIDTH_BIAS - p['B_IL'] - 1
    OUT_FL = WIDTH_OUT_DATA - p['OUT_IL'] - 1

    if layer_state == 4:  # maxpool 모드: bias 없음
        bias_ext = 0
    else:
        # bias extension: sign-extend and scale
        shift_amt = B_FL - EXT_FL
        bias_ext = bias >> shift_amt
        bias_ext = torch.nn.functional.pad(
            bias_ext.view(1), (WIDTH_BIAS - EXT_FL - p['B_IL'], 0), value=bias.sign().item()
        ).squeeze()

    bias_added = (data_extended + bias_ext).to(torch.int32)

    # ReLU 모드
    if layer_state in [1, 2, 3, 4, 5]:
        if bias_added < 0:
            return torch.zeros(1, dtype=torch.int8)
        max_val = (1 << (p['OUT_IL'] + EXT_FL)) - 1
        if bias_added > max_val:
            return torch.tensor([2 ** (WIDTH_OUT_DATA - 1) - 1], dtype=torch.int8)
        else:
            out = bias_added >> (EXT_FL - OUT_FL)
            return out.to(torch.int8)

    # no_relu 모드
    elif layer_state == 6:
        max_val = (1 << (p['OUT_IL'] + EXT_FL)) - 1
        min_val = -(1 << (p['OUT_IL'] + EXT_FL))
        if bias_added < min_val:
            return torch.tensor([-128], dtype=torch.int8)
        elif bias_added > max_val:
            return torch.tensor([127], dtype=torch.int8)
        else:
            out_shifted = bias_added >> (EXT_FL - OUT_FL)
            return out_shifted.to(torch.int8)

    # fallback
    return torch.zeros(1, dtype=torch.int8)


In [ ]:
params = {
    'WIDTH_IN_DATA': 24,
    'WIDTH_BIAS': 32,
    'WIDTH_BIAS_ADDED': 33,
    'WIDTH_OUT_DATA': 8,
    'layers': {
        'L1': {'IN_IL': 9, 'OUT_IL': 2, 'B_IL': 1},
        'L2': {'IN_IL': 9, 'OUT_IL': 2, 'B_IL': 1},
        'L3': {'IN_IL': 9, 'OUT_IL': 2, 'B_IL': 1},
        'L4': {'IN_IL': 9, 'OUT_IL': 2, 'B_IL': 1},
        'L5': {'IN_IL': 9, 'OUT_IL': 2, 'B_IL': 1},
        'L6': {'IN_IL': 9, 'OUT_IL': 2, 'B_IL': 1}
    }
}


In [9]:
def save_txt(tensor, file_name):
    import numpy as np
    assert tensor.dim()==2
    tensor=np.array(tensor).astype(int).astype(str).tolist()
    with open(file_name, "w") as file:
        for row in tensor:
            file.write(" ".join(row) + "\n")

In [10]:
# init layer classes
layer1_en = layer1()
layer2_en = layer2()
layer3_en = layer3()
layer4_en = layer4()
layer5_en = layer5()
layer6_en = layer6()
SRAM_write = init_SRAM_write()
weight_write = init_weight_write()

# Main FSM 
#IDLE
print("Process Started")

#S_SRAM_W
SRAM_write(input_img, pixel_mask)
weight_write()
reg_b, _, _ = linear_quantize_feature(reg_b, 8)
mem_w, _, _ = linear_quantize_feature(mem_w, 8)
mem_A = mem_A.to(torch.int8)
#Layer 1~6
layer1_en()
mem_B = mem_B.to(torch.int8)
save_txt(mem_B, 'after_layer1_memB.txt')
layer2_en()
mem_A = mem_A.to(torch.int8)
save_txt(mem_A, 'after_layer2_memA.txt')
layer3_en()
mem_B = mem_B.to(torch.int8)
save_txt(mem_B, 'after_layer3_memB.txt')
layer4_en()
mem_A = mem_A.to(torch.int8)
save_txt(mem_A, 'after_layer4_memA.txt')
layer5_en()
mem_B = mem_B.to(torch.int8)
save_txt(mem_B, 'after_layer5_memB.txt')
layer6_en()
mem_A = mem_A.to(torch.int8)
save_txt(mem_A, 'after_layer6_memA.txt')
#IDLE
print("Process Finished")

Process Started
memory A write done
weight write done
layer 1 done
layer 2 done
layer 3 done
torch.Size([1, 16, 100, 100]) torch.Size([1, 16, 100, 100]) torch.Size([1, 16, 100, 100])
torch.Size([1, 16, 50, 25]) torch.Size([1, 16, 50, 50]) torch.Size([1, 16, 50, 25])
layer 4 done
torch.Size([1, 16, 50, 25]) torch.Size([1, 16, 50, 50]) torch.Size([1, 16, 50, 25])
layer 5 done
torch.Size([1, 1, 50, 25]) torch.Size([1, 1, 50, 50]) torch.Size([1, 1, 50, 25])
layer 6 done
Process Finished


In [7]:
print(torch.load('layer1_R.pt'))

tensor([[[[    0.0000,     0.0000,     0.2479,     0.0234,     0.0000,
               0.0000,     0.2608,     0.0538,     0.0000,     0.0000,
               0.2547,     0.0433,     0.0000,     0.0000,     0.2511,
               0.0138,     0.0000,     0.0000,     0.2922,     0.0230,
               0.0000,     0.0000,     0.2443,     0.0201,     0.0000,
               0.0000,     0.2627,     0.0168,     0.0000,     0.0000,
               0.2425,     0.0130,     0.0000,     0.0000,     0.2949,
               0.0635,     0.0000,     0.0000,     0.2342,     0.0266,
               0.0000,     0.0000,     0.2399,     0.0057,     0.0000,
               0.0000,     0.2395,     0.0143,     0.0000,     0.0000,
               0.2198,     0.0283,     0.0000,     0.0000,     0.2917,
               0.0449,     0.0000,     0.0000,     0.2181,     0.0440,
               0.0000,     0.0000,     0.2451,     0.0413,     0.0000,
               0.0000,     0.2690,     0.0316,     0.0000,     0.0000,
      

In [13]:
def int_to_hex(int_tensor): #print the data in hexa format
    int_tensor=int_tensor.int()
    # Step 5: Convert to binary two's complement representation (optional for display)
    binary_tensor = [format(x & 0xFF, '08b') for x in int_tensor.view(-1).tolist()]
    hex_tensor=[ hex(int(binary_str[:4], 2))[2:].upper()+hex(int(binary_str[4:], 2))[2:].upper() for binary_str in binary_tensor]
    output = "\n".join("".join(hex_tensor[i * 16:(i + 1) * 16]) for i in range(31008)) #16 num, 210 lines -> 32 x 210

    return output

print(int_to_hex(mem_A))

00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
00000000000000000000000000000000
0000000000